
# Exercício Prático 1 — Diagnóstico e Correção de *Race Condition* em Python (**multiprocessing**)

Este notebook guia, passo a passo, a **reprodução**, o **diagnóstico** e a **correção** de uma *race condition* usando **processos** (`multiprocessing`),
evitando o efeito do **GIL** (Global Interpreter Lock) do CPython que pode mascarar problemas quando usamos `threading`.

> **Objetivo didático:** demonstrar uma condição de corrida “real” (com paralelismo efetivo) em uma atualização do tipo *read–modify–write*,
e corrigir o problema com um mecanismo de sincronização (Lock/mutex), discutindo custo e corretude.

---

## Por que `multiprocessing`?
- Em `threading` (CPython), o GIL pode reduzir muito o interleaving “perigoso”.
- Em `multiprocessing`, **cada processo tem seu próprio interpretador** → pode haver execução simultânea em múltiplos núcleos.
- Para ocorrer *race condition*, precisamos de **estado compartilhado** (memória compartilhada), que será nosso contador.

---

## Nota importante sobre `Value` e “proteção implícita”
`multiprocessing.Value` por padrão cria um **lock interno** (`lock=True`).  
Se você quer **reproduzir a corrida**, precisa **desligar** essa proteção: `lock=False`.



### Observação sobre execução em notebooks (Windows/macOS)

Em alguns ambientes, `multiprocessing` usa o método **spawn**, que pode exigir proteção com:

```python
if __name__ == "__main__":
    ...
```

Em Jupyter geralmente funciona, mas se você tiver problemas, rode este notebook como script `.py`
ou configure o método de start (Linux costuma usar `fork` por padrão):

```python
import multiprocessing as mp
mp.set_start_method("spawn", force=True)  # ou "fork" em Linux
```



## 1) Configuração do experimento
Vamos usar:
- `N_PROCS`: número de processos concorrentes
- `N_ITERS`: incrementos por processo
- `expected = N_PROCS × N_ITERS`

A expectativa é que, **sem sincronização**, o resultado possa ser **menor** que `expected` devido a *lost updates*.


In [ ]:

import multiprocessing as mp
import time
import statistics

N_PROCS = 8
N_ITERS = 200_000



## 2) Código base (com problema): contador compartilhado sem lock

O padrão `counter.value += 1` é uma operação composta (*read → add → write*).  
Quando múltiplos processos executam isso em paralelo no mesmo endereço compartilhado, pode ocorrer **perda de atualização** (*lost update*).

⚠️ Para garantir que **não existe proteção implícita**, criamos o `Value` com `lock=False`.


In [ ]:

def worker_race(counter, n_iters):
    for _ in range(n_iters):
        counter.value += 1  # NÃO é atômico (read-modify-write)

def run_race_condition(n_procs=N_PROCS, n_iters=N_ITERS):
    counter = mp.Value("i", 0, lock=False)

    procs = [mp.Process(target=worker_race, args=(counter, n_iters)) for _ in range(n_procs)]
    for p in procs: p.start()
    for p in procs: p.join()

    return counter.value



## 3) Tarefa 1 — Diagnóstico (rodar várias vezes e registrar)

Execute o experimento várias vezes e registre:
- mínimo, máximo, média
- quantas vezes ficou abaixo do esperado

> **Pergunta:** por que o resultado pode ser menor que o esperado?
Relacione com **interleaving**, operação **não atômica** e **ausência de exclusão mútua**.


In [ ]:

def sample_runs(fn, trials=10):
    results = []
    t0 = time.perf_counter()
    for _ in range(trials):
        results.append(fn())
    elapsed = time.perf_counter() - t0
    return results, elapsed

expected = N_PROCS * N_ITERS
results, elapsed = sample_runs(run_race_condition, trials=10)

print("Esperado:", expected)
print("Resultados:", results)
print("Mínimo :", min(results))
print("Máximo :", max(results))
print("Média  :", round(statistics.mean(results), 2))
print("Abaixo do esperado:", sum(r < expected for r in results), "de", len(results))
print("Tempo total:", round(elapsed, 3), "s")



## 4) Explicação formal do erro: *lost update*

O incremento `counter.value += 1` pode ser decomposto em:

1. **read**: ler `counter.value`
2. **increment**: somar 1 localmente
3. **write**: escrever o novo valor

Uma interleaving típica que perde atualização:

- Processo A lê 10  
- Processo B lê 10  
- Processo A escreve 11  
- Processo B escreve 11 (sobrescreve o valor de A)

✅ Dois incrementos “tentados”, mas só um persistiu.

Sem sincronização, não existe uma relação de ordenação que garanta exclusão mútua na seção crítica.



## 5) Tarefa 2 — Correção com Lock (mutex)

Agora vamos corrigir usando exclusão mútua.
Criamos um `mp.Lock()` e protegemos a seção crítica:

- adquirir lock → atualizar contador → liberar lock

Isso força apenas **um processo por vez** a executar o trecho crítico.


In [ ]:

def worker_with_lock(counter, lock, n_iters):
    for _ in range(n_iters):
        with lock:
            counter.value += 1

def run_with_lock(n_procs=N_PROCS, n_iters=N_ITERS):
    counter = mp.Value("i", 0, lock=False)  # lock externo, explícito
    lock = mp.Lock()

    procs = [mp.Process(target=worker_with_lock, args=(counter, lock, n_iters)) for _ in range(n_procs)]
    for p in procs: p.start()
    for p in procs: p.join()

    return counter.value


In [ ]:

results_lock, elapsed_lock = sample_runs(run_with_lock, trials=10)

print("Esperado:", expected)
print("Resultados:", results_lock)
print("Mínimo :", min(results_lock))
print("Máximo :", max(results_lock))
print("Média  :", round(statistics.mean(results_lock), 2))
print("Abaixo do esperado:", sum(r < expected for r in results_lock), "de", len(results_lock))
print("Tempo total:", round(elapsed_lock, 3), "s")



## 6) Comparação de desempenho (antes × depois)

Sincronização melhora a corretude, mas adiciona overhead.
Vamos comparar tempos médios (race vs lock).


In [ ]:

def bench(fn, trials=5):
    outs = []
    times = []
    for _ in range(trials):
        t0 = time.perf_counter()
        outs.append(fn())
        times.append(time.perf_counter() - t0)
    return outs, times

outs_race, times_race = bench(run_race_condition, trials=5)
outs_lock2, times_lock2 = bench(run_with_lock, trials=5)

print("=== Sem lock (race) ===")
print("Saídas:", outs_race)
print("Tempo médio:", round(statistics.mean(times_race), 4), "s")

print("\n=== Com lock ===")
print("Saídas:", outs_lock2)
print("Tempo médio:", round(statistics.mean(times_lock2), 4), "s")



## 7) O que entregar (Checklist)

- [ ] Valor esperado `N_PROCS × N_ITERS`
- [ ] Evidência experimental (10+ execuções) **sem lock** mostrando variação/erro
- [ ] Explicação formal do *lost update* (read–modify–write + interleaving)
- [ ] Implementação da correção com `mp.Lock()`
- [ ] Evidência experimental **com lock** (sempre igual ao esperado)
- [ ] Comparação de desempenho (tempo médio antes/depois)



## 8) Extensão (opcional): mostrando a proteção implícita do `Value(lock=True)`

Aqui comparamos com `lock=True`, que **protege** a atualização do `.value` via lock interno.


In [ ]:

def run_with_value_internal_lock(n_procs=N_PROCS, n_iters=N_ITERS):
    counter = mp.Value("i", 0, lock=True)  # lock interno do Value

    def worker(counter, n_iters):
        for _ in range(n_iters):
            counter.value += 1  # protegido

    procs = [mp.Process(target=worker, args=(counter, n_iters)) for _ in range(n_procs)]
    for p in procs: p.start()
    for p in procs: p.join()
    return counter.value

print("Com lock interno do Value:", run_with_value_internal_lock(), "| Esperado:", expected)
